# Per-Hedge Metrics
Computes mean hedge accuracy and hedge detection rate from saved ground truth and predictions.
Optionally restricts evaluation to hedgerows within agricultural zones using RPG masks.

In [ ]:
import os
import sys
sys.path.insert(0, os.path.abspath(".."))

import torch
import numpy as np
import geopandas as gpd
import pandas as pd
from tqdm import tqdm
from dotenv import load_dotenv

from hedgementation_utils.io.io_manager import HedgementationIOManager
from hedgementation_utils.metrics.seg_meter import SegMeter as HedgeSegMeter

load_dotenv("../.env")

DATASET_ROOT = DATASET_ROOT = "/scratch/nathan/data/hedgementation_1.2"#os.environ["DATASET_ROOT"]
#os.environ["DATASET_ROOT"]
MODELS_DIR   = os.path.join("..", os.environ.get("MODELS_DIR", "models"))

print(f"DATASET_ROOT : {DATASET_ROOT}")
print(f"MODELS_DIR   : {os.path.abspath(MODELS_DIR)}")

In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────
# List every model you want to evaluate.  Each entry must have a
# full_dataset_predictions.pt inside MODELS_DIR/<name>/
MODEL_NAMES = [
    "UTAE_hedge_1.2_base",
]

# Threshold used for hedge detection rate  (a hedge is "detected" if its
# per-hedge recall >= this value)
DETECTION_THRESHOLD = 0.3

# ── Agricultural mask (optional) ─────────────────────────────────────────────
# Subdirectory of DATASET_ROOT that contains per-patch agricultural mask files.
# The IO manager looks for files named <MASK_FILE_PATTERN>.<MASK_FILE_EXT>
# (e.g. rpg_10042.npy).  Hedgerow pixels that fall outside the agricultural
# zone are zeroed out of the y_id raster before per-hedge metrics are computed,
# so only hedgerows with at least one pixel inside the agricultural zone are
# included in the final statistics.
# Set MASK_SUBDIR to None to skip masking and evaluate all hedgerows.
MASK_SUBDIR       = os.environ.get("RPG_MASK_SUBDIR", None)  # e.g. "rpg_masks_buffered_10m"
MASK_FILE_EXT     = "npy"    # "npy" or "tif"
MASK_FILE_PATTERN = "rpg_{}" # must match the file naming convention in MASK_SUBDIR

print(f"Agricultural masking : {'ENABLED  —  ' + str(MASK_SUBDIR) if MASK_SUBDIR else 'DISABLED'}")

In [ ]:
# ── Load metadata and build patch-ID → tensor-index mapping ─────────────────
metadata = gpd.read_file(os.path.join(DATASET_ROOT, "metadata.geojson"))
id_to_index = {
    int(pid): idx
    for idx, pid in enumerate(metadata["ID_PATCH"])
}
patch_ids = list(metadata["ID_PATCH"].astype(int))
print(f"Loaded metadata: {len(metadata)} patches")

In [ ]:
# ── Load ground truth ────────────────────────────────────────────────────────
gt_path = os.path.join(MODELS_DIR, "ground_truth_cached.pt")
ground_truth = torch.load(gt_path, weights_only=False)
print(f"Ground truth shape : {ground_truth.shape}  dtype={ground_truth.dtype}")

In [ ]:
# ── Load predictions for each model ─────────────────────────────────────────
prediction_mapping = {}
for name in MODEL_NAMES:
    pred_path = os.path.join(MODELS_DIR, name, "full_dataset_predictions.pt")
    if not os.path.exists(pred_path):
        print(f"  [SKIP] {name} — predictions not found")
        continue
    prediction_mapping[name] = torch.load(pred_path, weights_only=False)
    print(f"  [OK]   {name}  {prediction_mapping[name].shape}")

print(f"\nLoaded {len(prediction_mapping)} model(s)")

In [ ]:
# ── Per-hedge helper functions ───────────────────────────────────────────────
def compute_per_hedge_accuracies(
    predictions: torch.Tensor,   # (H, W)  int64
    y_id_raster: torch.Tensor,   # (H, W)  int64, hedge IDs start at 1
) -> torch.Tensor:
    """Return a 1-D tensor of per-hedge recall values for this patch."""
    num_hedgerows = int(y_id_raster.max().item())
    if num_hedgerows == 0:
        return torch.tensor([])

    scores = []
    for hedge_id in range(1, num_hedgerows + 1):
        hedge_mask = (y_id_raster == hedge_id).long()
        if hedge_mask.sum() == 0:      # hedge fully masked out — skip
            continue
        meter = HedgeSegMeter(num_classes=2)
        meter.update(predictions, hedge_mask)
        scores.append(meter.ask_metrics(cl=1)["acc"])

    return torch.tensor(scores, dtype=torch.float32)


def mean_hedge_accuracy(per_hedge_accs: torch.Tensor) -> float:
    return per_hedge_accs.mean().item() if len(per_hedge_accs) > 0 else float("nan")


def hedge_detection_rate(per_hedge_accs: torch.Tensor,
                         threshold: float = DETECTION_THRESHOLD) -> float:
    if len(per_hedge_accs) == 0:
        return float("nan")
    return (per_hedge_accs >= threshold).float().mean().item()

In [ ]:
# ── Build IO manager ─────────────────────────────────────────────────────────
# When MASK_SUBDIR is set the manager is configured to load RPG masks from that
# subdirectory; otherwise masking is simply skipped in the loop below.
io_manager_kwargs = dict(dataset_root=DATASET_ROOT)
if MASK_SUBDIR:
    io_manager_kwargs["rpg_mask_dir"]  = MASK_SUBDIR
    io_manager_kwargs["rpg_pattern"]   = MASK_FILE_PATTERN
    io_manager_kwargs["rpg_file_type"] = MASK_FILE_EXT

io_manager = HedgementationIOManager(**io_manager_kwargs)

In [ ]:
# ── Compute per-hedge metrics for all models ─────────────────────────────────
results = {}   # model_name -> {"mean_hedge_accuracy": float, "hedge_detection_rate": float}

for model_name, preds in prediction_mapping.items():
    print(f"\n{'─'*60}")
    print(f"Model: {model_name}")

    all_per_hedge_accs = []

    for patch_id in tqdm(patch_ids, desc="  patches", leave=False):
        idx = id_to_index[patch_id]
        patch_preds = preds[idx]           # (128, 128)

        try:
            y_id_np = io_manager.load_y_id(identifier=patch_id)
            y_id    = torch.from_numpy(y_id_np.astype(np.int64)).squeeze()
        except Exception:
            continue   # patch has no y_id file — skip

        if y_id.max() == 0:
            continue   # no labelled hedgerows in this patch

        # ── Apply agricultural mask (optional) ────────────────────────────
        if MASK_SUBDIR:
            try:
                ag_mask_np = io_manager.load_rpg_mask(identifier=patch_id)
                ag_mask    = torch.from_numpy(ag_mask_np.astype(bool)).squeeze()
                # Zero out hedge IDs that fall outside the agricultural zone.
                # Hedgerows whose remaining pixels are all zero will be skipped
                # inside compute_per_hedge_accuracies.
                y_id = y_id * ag_mask.long()
            except Exception:
                pass   # no mask file for this patch — evaluate all hedgerows

        patch_accs = compute_per_hedge_accuracies(patch_preds, y_id)
        if len(patch_accs) > 0:
            all_per_hedge_accs.append(patch_accs)

    if not all_per_hedge_accs:
        print("  No y_id data found — skipping")
        continue

    pooled = torch.cat(all_per_hedge_accs)
    mha    = mean_hedge_accuracy(pooled)
    hdr    = hedge_detection_rate(pooled)

    results[model_name] = {
        "mean_hedge_accuracy" : mha,
        "hedge_detection_rate": hdr,
        "num_hedgerows"       : len(pooled),
    }

    print(f"  Mean hedge accuracy  : {mha:.4f}")
    print(f"  Hedge detection rate : {hdr:.4f}  (threshold={DETECTION_THRESHOLD})")
    print(f"  Total hedgerows      : {len(pooled)}")

In [ ]:
# ── Summary table ────────────────────────────────────────────────────────────
if results:
    df = pd.DataFrame(results).T.rename_axis("model").reset_index()
    df = df.sort_values("mean_hedge_accuracy", ascending=False).reset_index(drop=True)
    display(df)
else:
    print("No results — check that y_id rasters exist under DATASET_ROOT/y_id/")